In [ ]:
!pip install tensorflow opencv-python mediapipe scikit-learn matplotlib
# Verify installations
import tensorflow as tf
import cv2 as cv2
import mediapipe as mp
from sklearn import datasets
import matplotlib.pyplot as plt
import time
import os
import numpy as np
print("All libraries imported successfully.")
# NOTE: always run this when freshly opening this/working on it

In [ ]:
mp_holistic = mp.solutions.holistic # Holistic models
mp_drawing = mp.solutions.drawing_utils #drawing utils
print("holistics drawing is ready") # Confirms it

In [ ]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) #From BGR (Blue, Green, Red) to RGB
    image.flags.writeable = False #Restricts the image so it wouldnt be modified
    results = model.process(image) # Prediciton
    image.flags.writeable = True #Allows image to be modified
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) #From RGB to BGR (Blue, Green, Red)
    return image, results

In [ ]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION) # this is face connections replaced by attribute "facemesh"
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) # draws pose connections
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) # draws left hand 
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)# draws right hand

In [ ]:
def draw_styled_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION,
        mp_drawing.DrawingSpec(color=(80, 110, 10), thickness=1, circle_radius=1),  # the landmark aka circle color
        mp_drawing.DrawingSpec(color=(80, 0, 121), thickness=1, circle_radius=1)  # the connection aka line color
    )  # this is face connections replaced by attribute "facemesh"

    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(80, 22, 10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80, 44, 121), thickness=2, circle_radius=2)
    )  # draws pose connections

    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(121, 22, 76), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(121, 44, 250), thickness=2, circle_radius=2)
    )  # draws left hand 

    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(80, 110, 10), thickness=2, circle_radius=4),
        mp_drawing.DrawingSpec(color=(80, 0, 121), thickness=2, circle_radius=2)
    )  # draws right hand NOTE: FIX THIS ISSUE WITH COLOR CUSTOMISATION LATER!!!!!!!(fixed)


In [ ]:
import cv2

cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        print(results)
        
        # draws landsmarks(key points)
        draw_styled_landmarks(image, results)
        
        # Show to screen
        cv2.imshow('Mizo Frame', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('x'):
            break
    cap.release()
    cv2.destroyAllWindows()

In [ ]:
len(results.left_hand_landmarks.landmark) #gets results here in this case finds the amount of left hand

In [ ]:
results

In [ ]:
draw_landmarks(frame, results)

In [ ]:
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))  # note this changes it from bgr to rgb cuz it looks funky :)

In [ ]:
pose = [] 
for res in results.pose_landmarks.landmark:
    test = np.array([res.x, res.y, res.z, res.visibility])
    pose.append(test)

In [ ]:
pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132)
face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404) # error if statement in case the face is not in frame
lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3) # error if statement in case the left hand is not in frame
rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)# error if statement in case the right hand is not in frame

In [ ]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z, res.visibility] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(132)
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(1404) # error if statement in case the face is not in frame
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3) # error if statement in case the left hand is not in frame
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)# error if statement in case the right hand is not in frame
    return np.concatenate([pose, face, lh, rh])# merges all the points together

In [ ]:
# where all the data from 3. goes
DATA_PATH = os.path.join("MP_Data")
 
# actions to try to detect/recognise 
actions = np.array(["yes","no","hello","goodbye"]) # NOTE: ADD more to make the ai more better and useful 
# 30 videos worth of data NOTE: you can make it bigger for more accuracy
no_sequences = 60
# 30 frames per video NOTE: you can change it to 60 but would need to change the data shape from 30,1692 to 60,1692
sequence_length = 60

In [ ]:
for action in actions:
    for sequence in range(no_sequences):
        try:
            os.makedirs(os.path.join(DATA_PATH, action, str(sequence)))
        except FileExistsError:
            pass

In [ ]:
import cv2

cap = cv2.VideoCapture(0)

# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:

    # Loops through the actions
    for action in actions:
        # Loops through no_sequences
        for sequence in range(no_sequences):
            # Loops through sequence_length
            for frame_num in range(sequence_length):
                
                # Read feed
                ret, frame = cap.read()
                if not ret:
                print("Camera error, stopping collection")
                cap.release()
                cv2.destroyAllWindows()
                break #catches frame drop if any issues with camera, it will terminate the program
                
                # Make detections
                image, results = mediapipe_detection(frame, holistic)
                print(results)
                
                # Draw landmarks (key points)
                draw_styled_landmarks(image, results)

                # Applying Collection delay to collect action easier
                if frame_num == 0:
                    cv2.putText(image, "STARTING COLLECTION", (120, 200),
                                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 4, cv2.LINE_AA)
                    cv2.putText(image, "Collecting frames for {} Video Number {}".format(action, sequence), (15, 12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    cv2.imshow('Mizo Frame', image)
                    cv2.waitKey(3000)
                else:
                    cv2.putText(image, "Collecting frames for {} Video Number {}".format(action, sequence), (15, 12),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)
                    # Show to screen
                    cv2.imshow('Mizo Frame', image)

                # Exports keypoints to file directory
                keypoints = extract_keypoints(results)
                npy_path = os.path.join(DATA_PATH, action, str(sequence), str(frame_num))
                np.save(npy_path, keypoints)

                # Break gracefully
                if cv2.waitKey(10) & 0xFF == ord('x'):
                    break
                     
    cap.release()
    cv2.destroyAllWindows()


In [ ]:
label_map = {label:num for num, label in enumerate(actions)}

In [ ]:
sequences, label = [], []
for action in actions:
    for sequence in range(no_sequences):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action , str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        label.append(label_map[action])

In [ ]:
X= np.array(sequences)

In [ ]:
from sklearn.preprocessing import StandardScaler
import pickle

nsamples, nframes, nfeatures = X.shape #shapes into (180,60,1692)
X_reshaped = X.reshape(-1, nfeatures) # flattens to (10800, 1692) so scaler works on features

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped)  # learns mean/std from training data
X = X_scaled.reshape(nsamples, nframes, nfeatures)  # reshape back to (180,60,1692)

# Save the scaler — you MUST use this same scaler at inference time
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

print(f"Data shape: {X.shape}")
print("Scaler saved.")

In [ ]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(label).astype(int)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # Training and testing partition NOTE: Change test_size based on amount of datapoints
# random state makes the shuffles the same in order to know it wasnt luck but an improvement
print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import TensorBoard
print("LSTM imported")

In [ ]:
from tensorflow.keras.callbacks import TensorBoard
import datetime

run_name = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")#formatted as Year,Month,Day,Hour,Minute,Second
# Defines the path for the "Logs" folder and the metadata associated
log_dir = os.path.join("Logs", run_name)

# Ensure that the "Logs" folder exists
if not os.path.exists(log_dir):
    os.makedirs(log_dir)

# Initialize the TensorBoard callback with the log directory
tb_callback = TensorBoard(log_dir=log_dir,histogram_freq=1,write_graph=True,update_freq="epoch")
# tb_callback has been updated to show weight and bias distribution per epoch session,visualising model graph and logs frequency per epoch
# You can now use tb_callback in model training

early_stop = EarlyStopping(monitor="val_loss",patience=50,restore_best_weights=True)
# watches validation loss and stops if no improvement for 50 epoch and saves the weights based on the highest epoch accuracy
print(f"Logging to: {log_dir}")

In [ ]:
model = Sequential()
model.add(LSTM(64, return_sequences=True, activation="tanh", input_shape=(60,1692)))
model.add(Dropout(0.2))
model.add(LSTM(128, return_sequences=True, activation="tanh"))
model.add(Dropout(0.2))
model.add(LSTM(64, return_sequences=False, activation="tanh"))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(32, activation="relu"))
model.add(Dense(actions.shape[0], activation="softmax"))

In [ ]:
model.compile(optimizer="Adam", loss="categorical_crossentropy", metrics=["categorical_accuracy"])  #metrics is the variable that concludes the model accureate to what it is displaying

In [ ]:
model.summary()

In [ ]:
model.fit(X_train, y_train, epochs=2000, validation_data=X_test, y_test), callbacks=[tb_callback, early_stop]) #NOTE use tensorboard web to see data through cmd "tensorboard --logdir=logs"
# validation curves in Tensorboard for histogram and graphs

8.Making Predictions

In [ ]:
model.predict(X_test)

In [ ]:
actions[np.argmax(res[0])]

In [ ]:
actions[np.argmax(y_test[0])]

9.Saving the Data(Weights)

In [ ]:
model.save("Mizo.h5") # Save the TF Model

In [ ]:
del model #NOTE: USE IT IF MODEL IS UNSTABLE AND/OR UNUSABLE // AS A LAST RESORT, REBUILD THE MODEL

In [ ]:
model.load_weights("Mizo.h5") # ONLY Recovers the weight file if user accidently deletes it

In [ ]:
from tensorflow.keras.models import load_model

model= load_model("Mizo.h5") #Recovers the model file if the user accidently deletes it

10.Evaluation using Confusion Matrix and Accuracy

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix, accuracy_score

In [ ]:
yhat = model.predict(X_test)

In [ ]:
ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

In [ ]:
multilabel_confusion_matrix(ytrue, yhat) #NOTE refer to the confusion matrix with false negatives and true positives

In [ ]:
accuracy_score(ytrue, yhat)

11. TESTING IN REAL TIME 

In [ ]:
colors = [(245,117,16), (117,245,16), (16,117,245)]
def prob_viz(res, actions, input_frame, colors):
    output_frame = input_frame.copy()
    for num, prob in enumerate(res):
        cv2.rectangle(output_frame, (0,60+num*40), (int(prob*100), 90+num*40), colors[num], -1)
        cv2.putText(output_frame, actions[num], (0, 85+num*40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2, cv2.LINE_AA)

    return output_frame

In [ ]:
# New detections
import pickle
with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)
    
sequence = []  # collects the 60 frames for more recognisable and smoother input
sentence = [] 
predictions = []  # predicts the action during transition
threshold = 0.5  # optimize the threshold for accuracy

cap = cv2.VideoCapture(0)
# Set mediapipe model 
with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
    while cap.isOpened():

        # Read feed
        ret, frame = cap.read()
        
        if not ret:
            print("Failed to grab frame")
            break

        # Make detections
        image, results = mediapipe_detection(frame, holistic)
        
        # Check if the results contain landmarks
        if not results.pose_landmarks:
            print("No landmarks detected")
            continue
        
        # Draw landmarks (key points)
        draw_styled_landmarks(image, results)

        # Prediction logic
        keypoints = extract_keypoints(results)
        keypoints = scaler.transform(keypoints.reshape(1,-1)).flatten()
        sequence.append(keypoints)
        sequence = sequence[-60:]

        if len(sequence) == 60:
            res = model.predict(np.expand_dims(sequence, axis=0))[0]
            print(f"Raw model prediction: {res}")

            predictions.append(np.argmax(res)) #actually makes the logic be executable 
            predictions = predictions[-10:] #prevents it from exponentially growing predictions and adds a cap
            
                
                # Visualization logic
                if len(predictions) > 0 and len(np.unique(predictions)) == 1:
                    if res[np.argmax(res)] > threshold:
                        
                        if len(sentence) > 0:
                            if predicted_action != sentence[-1]:
                                sentence.append(predicted_action)
                        else:
                            sentence.append(predicted_action)

                sentence = sentence[-5:]  # Keep only the last 5 predictions
                print(f"Current sentence: {sentence}")
                
                image = prob_viz(res, actions, image, colors)

                cv2.rectangle(image, (0, 0), (640, 40), (245, 117, 16), -1)
                cv2.putText(image, " ".join(sentence), (3, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)  # Determines the box on the top left
        
        # Show to screen
        cv2.imshow('Mizo Frame', image)

        # Break gracefully
        if cv2.waitKey(10) & 0xFF == ord('x'):
            break

cap.release()
cv2.destroyAllWindows()
